# Assignment: When Chat Systems Break - A Realistic Sharding Simulation


---
## 📅 Day 7 (7 April): Channel-Based Sharding — Second Wrong Decision

In [ ]:
import random


class Message:
    def __init__(self, user_id, channel_id, content):
        self.user_id = user_id
        self.channel_id = channel_id
        self.content = content


class Shard:
    def __init__(self, shard_id):
        self.id = shard_id
        self.messages = []

    def store(self, message):
        self.messages.append(message)

    def count(self):
        return len(self.messages)


class ShardManager:
    def __init__(self, num_shards):
        self.shards = [Shard(i) for i in range(num_shards)]

    def send_message(self, message):
        # will be implemented by subclasses
        pass

    def print_stats(self):
        total = sum(s.count() for s in self.shards)
        print(f"{'Shard':<10} {'Messages':<15} {'% of Total':<10}")
        print('-' * 35)
        for shard in self.shards:
            pct = round((shard.count() / total * 100), 1) if total > 0 else 0
            print(f"Shard {shard.id:<5} {shard.count():<15} {pct}%")
        print(f"{'Total':<10} {total}")


In [ ]:
# =============================================
# DAY 7: Channel-Based Sharding
# Logic: channel_id % num_shards → decides shard
# Problem: viral channel → one shard gets all traffic
# =============================================

class ChannelShardManager(ShardManager):

    def get_shard(self, channel_id):
        return self.shards[channel_id % len(self.shards)]

    def send_message(self, message):
        shard = self.get_shard(message.channel_id)
        shard.store(message)


# --- Simulate: channel 3 (Cricket Final) goes viral ---
# --- 80% of all messages go to channel 3 ---

print("=== Channel-Based Sharding Simulation ===")
print("Scenario: Channel 3 (Cricket Final) goes viral")
print("          80% of messages go to channel 3")
print("          20% spread across other channels")
print()

ch_manager = ChannelShardManager(num_shards=3)

total_messages = 10000
viral_messages = int(total_messages * 0.8)  # 8000 to channel 3
normal_messages = total_messages - viral_messages  # 2000 spread

# 8000 messages to the viral channel (channel_id = 3)
for i in range(viral_messages):
    msg = Message(user_id=random.randint(1, 50000), channel_id=3, content="INDIA WINS!")
    ch_manager.send_message(msg)

# 2000 messages spread across other channels (not 3)
other_channels = [1, 2, 4, 5, 6, 7, 8, 9, 10]
for i in range(normal_messages):
    ch_id = random.choice(other_channels)
    msg = Message(user_id=random.randint(1, 50000), channel_id=ch_id, content="hello")
    ch_manager.send_message(msg)

print("--- Messages per Shard ---")
ch_manager.print_stats()
print()
print("--- Observation ---")
print(f"channel_id 3 % 3 = 0 → ALL viral messages go to Shard 0!")
print("Shard 0 = OVERLOADED (hotspot problem)")
print("Shard 1 and Shard 2 = mostly idle")
print()
print("--- Comparison with User-Based Sharding ---")
print("User-based: 1 active user floods 1 shard")
print("Channel-based: 1 viral channel floods 1 shard")
print("Both strategies fail during spikes — just for different reasons!")